# ActiveSAM · ADE20K 论文结果复现

先将本目录部署文件推送到你的 GitHub 分支，再只用这一份笔记本启动。
选择支持原生 BF16、显存足够的 GPU，按顺序运行。
默认执行 **20 张流程检查 → 2,000 张默认配置 → 2,000 张关闭 EPG 对照**。
论文 v2 表 10 的参考 mIoU 为 **40.03 / 40.05**。

模型构建、提示扩展、EPG 和 ECD 继续使用作者源码；配置直接读取作者文件。
图像保持原尺寸，沿用作者 336/1008 分辨率处理和 32 条提示分桶。
评测仅补充 ADE20K 标签映射、指标累计和结果导出，避免在 Colab 编译 MMCV。
本笔记本的 FPS 包含同步和标签传回 CPU，不能直接对照论文的 Runner FPS。

**本次精简：** 删除重复哈希、源码审计、归档清单、A/B 测试、文件夹转 ZIP 缓存及额外样例导出；
保留 ZIP 本地复用、CPU 输入预取、可中断推理和单文件断点。
全部可调参数在第 1、2 节，代码默认隐藏；点击“显示代码”可展开。
该版本不接续旧归档格式；首次使用请新建结果目录。

[论文](https://arxiv.org/abs/2606.16996) ·
[作者仓库](https://github.com/VILA-Lab/ActiveSAM)

## 1. 路径参数

**云盘 ZIP 路径是下面的 `DATASET_PATH`。** 支持验证集专用 ZIP 和官方完整 ZIP，
包内需有 `images/validation`、`annotations/validation`，图像和原始标注各 2,000 张。
ZIP 和权重只读取，不改写源文件。同一运行时重跑会复用本地副本。

`RESUME_RUN_DIRECTORY` 首次留空；续跑填上次打印的结果总目录。
WordNet 可使用已有的 `corpora/wordnet.zip` 或文件夹，留空则下载。

In [ ]:
# @title 路径参数：DATASET_PATH 为云盘 ZIP { display-mode: "form" }
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DATASET_PATH = "/content/drive/MyDrive/datasets/ADE20K/ADE20K.zip"  # @param {type:"string"}
WEIGHTS_PATH = "/content/drive/MyDrive/models/sam3/sam3.pt"  # @param {type:"string"}
OUTPUT_ROOT = "/content/drive/MyDrive/results/ActiveSAM_reproduction"  # @param {type:"string"}
WORDNET_DIR = "/content/drive/MyDrive/datasets/nltk_data"  # @param {type:"string"}
RESUME_RUN_DIRECTORY = ""  # @param {type:"string"}
WORK_DIR = "/content/activesam"  # @param {type:"string"}

DATASET_PATH, WEIGHTS_PATH, OUTPUT_ROOT, WORK_DIR = map(Path, (DATASET_PATH, WEIGHTS_PATH, OUTPUT_ROOT, WORK_DIR))
WORDNET_DIR = Path(WORDNET_DIR) if WORDNET_DIR else None
if DATASET_PATH.suffix.lower() != ".zip" or not DATASET_PATH.is_file():
    raise FileNotFoundError(f"请填写已有的数据集 ZIP：{DATASET_PATH}")
if not WEIGHTS_PATH.is_file():
    raise FileNotFoundError(f"找不到 SAM 3 权重：{WEIGHTS_PATH}")

Mounted at /content/drive


## 2. 来源与运行参数

默认从你的 `xheihex/ActiveSAM` 仓库下载，分支名为 **`colab-deployment`**（连字符），
代码目录为 `colab_deployment/`（下划线）。`GIT_REF` 也可填写固定提交 SHA。
续跑应保持数据、权重、提交与运行环境一致。

CPU 默认用 2 个线程提前读图，最多预取 4 张；0 表示串行输入。
GPU 由一个工作线程逐张调用作者模型，避免作者共享解码器状态的并发冲突。

In [ ]:
# @title GitHub 来源与运行参数 { display-mode: "form" }
GITHUB_URL = "https://github.com/xheihex/ActiveSAM.git"  # @param {type:"string"}
GIT_REF = "colab-deployment"  # @param {type:"string"}
RUN_NO_EPG = True  # @param {type:"boolean"}
SMOKE_IMAGES = 20  # @param {type:"integer"}
SEED = 1  # @param {type:"integer"}
CHECKPOINT_EVERY = 20  # @param {type:"integer"}
INFERENCE_PREFETCH_WORKERS = 2  # @param {type:"integer"}

if not 1 <= SMOKE_IMAGES <= 2000 or CHECKPOINT_EVERY < 1 or INFERENCE_PREFETCH_WORKERS < 0:
    raise ValueError("请检查流程检查张数、断点间隔和输入线程数。")

In [ ]:
# @title 下载所选分支 { display-mode: "form" }
import os
import subprocess
import sys

if "predictor" in globals():
    predictor.close(wait=False)
    del predictor


def checkout_repository(repository_url, git_ref, root):
    repo = Path(root) / "ActiveSAM"
    repo.mkdir(parents=True, exist_ok=True)
    environment = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1", GIT_TERMINAL_PROMPT="0")

    def git(*arguments):
        subprocess.run(["git", "-C", str(repo), *arguments], env=environment, check=True)

    if not (repo / ".git").exists():
        git("init", "--quiet")
        git("remote", "add", "origin", repository_url)
    else:
        git("remote", "set-url", "origin", repository_url)
    git("fetch", "--depth=1", "--", "origin", git_ref)
    git("checkout", "--quiet", "--detach", "FETCH_HEAD")
    commit = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
    return repo, commit


REPO_DIR, SOURCE_COMMIT = checkout_repository(GITHUB_URL, GIT_REF, WORK_DIR)
print("源码提交：", SOURCE_COMMIT)

源码提交： 2ec7f26ef1678599590f09b9bcba05b4988e783d


## 3. 安装依赖与 GPU 检查

复用 Colab 的 PyTorch / torchvision。缺少原生 BF16 的 GPU 会停止，
不自动更换精度或降低分辨率。若 Colab 要求重启，重启后从第 1 节运行。

In [ ]:
# @title 安装轻量依赖并加载部署工具 { display-mode: "form" }
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--prefer-binary",
        "-r",
        str(REPO_DIR / "colab_deployment/requirements_reproduction.txt"),
    ],
    check=True,
)
import gc
import random

import numpy as np
import torch
from tqdm.auto import tqdm

if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported(including_emulation=False):
    raise RuntimeError("请选择支持原生 BF16、显存足够的 GPU。")
if "model" in globals():
    del model
gc.collect()
torch.cuda.empty_cache()
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False

old_repo = globals().get("IMPORTED_REPO_DIR")
if old_repo and str(old_repo) in sys.path:
    sys.path.remove(str(old_repo))
for name in tuple(sys.modules):
    if name.split(".", 1)[0] in {"colab_deployment", "sam3"}:
        del sys.modules[name]
sys.path.insert(0, str(REPO_DIR))
from colab_deployment import reproduce as rep

IMPORTED_REPO_DIR = REPO_DIR
print(
    "PyTorch",
    torch.__version__,
    "CUDA",
    torch.version.cuda,
    "GPU",
    torch.cuda.get_device_name(),
)

PyTorch 2.14.1+cu130 CUDA 13.0 GPU NVIDIA L4


## 4. 准备 ZIP、权重和 WordNet

ZIP 先复制到本地，只解压验证集。文件名和数量必须符合完整验证集，
标注采用原始 ID：0 忽略，1..150 映射为模型的 0..149。
本地复用只比较源路径、大小和修改时间，不再逐文件计算哈希。

In [ ]:
# @title 准备输入并选择结果目录 { display-mode: "form" }
import shutil
from datetime import datetime

import nltk

LOCAL_CHECKPOINT = WORK_DIR / "sam3.pt"
rep.copy_local(WEIGHTS_PATH, LOCAL_CHECKPOINT)
ENTRIES = rep.prepare_ade20k(DATASET_PATH, WORK_DIR / "ADE20K", progress=tqdm)
LOCAL_WORDNET = WORK_DIR / "nltk_data"
(LOCAL_WORDNET / "corpora").mkdir(parents=True, exist_ok=True)
if WORDNET_DIR and (WORDNET_DIR / "corpora/wordnet.zip").is_file():
    rep.copy_local(WORDNET_DIR / "corpora/wordnet.zip", LOCAL_WORDNET / "corpora/wordnet.zip")
elif WORDNET_DIR and (WORDNET_DIR / "corpora/wordnet").is_dir():
    shutil.copytree(
        WORDNET_DIR / "corpora/wordnet",
        LOCAL_WORDNET / "corpora/wordnet",
        dirs_exist_ok=True,
    )
elif not (LOCAL_WORDNET / "corpora/wordnet.zip").exists():
    nltk.download("wordnet", download_dir=str(LOCAL_WORDNET), raise_on_error=True)
nltk.data.path.insert(0, str(LOCAL_WORDNET))

if RESUME_RUN_DIRECTORY:
    RUN_DIRECTORY = Path(RESUME_RUN_DIRECTORY)
    if not RUN_DIRECTORY.is_dir():
        raise FileNotFoundError("恢复目录不存在，请填写上次打印的结果总目录。")
else:
    RUN_DIRECTORY = OUTPUT_ROOT / datetime.now().strftime("%Y%m%d_%H%M%S")
    RUN_DIRECTORY.mkdir(parents=True, exist_ok=False)
INPUT_SETTINGS = {
    "dataset": rep.source_info(DATASET_PATH),
    "weights": rep.source_info(WEIGHTS_PATH),
    "seed": SEED,
}
print("结果总目录（恢复时填写此路径）：", RUN_DIRECTORY)

复制到本地： sam3.pt
复制到本地： ADE20K.zip


准备 ADE20K 验证集:   0%|          | 0/4000 [00:00<?, ?文件/s]

复制到本地： wordnet.zip
结果总目录（恢复时填写此路径）： /content/drive/MyDrive/results/ActiveSAM_reproduction/20261121_111812


## 5. 加载作者模型

使用原有单图适配器加载作者 `ActiveSAM` 类，读取作者基础配置与 ADE20K 配置。
推理直接调用作者的 `class_maps()` 和 `decode()`，不重写提示扩展、EPG 或 ECD。

In [ ]:
# @title 加载模型并定义评测入口 { display-mode: "form" }
if "predictor" in globals():
    predictor.close(wait=False)
    del predictor
if "model" in globals():
    del model
gc.collect()
torch.cuda.empty_cache()
os.environ["HF_HUB_OFFLINE"] = "1"
from colab_deployment.adapter import build_model

model_config, DEFAULT_CONFIG = rep.load_model_config(REPO_DIR, LOCAL_CHECKPOINT)
model, _ = build_model(model_config)
model.requires_grad_(False)
predictor = rep.InterruptiblePredictor(rep.TorchPredictor(model))
CLASS_NAMES = [line.split(",")[0].strip() for line in (REPO_DIR / "configs/cls_ade20k.txt").read_text().splitlines()]
if not (RUN_DIRECTORY / "run.json").exists():
    rep.write_json(
        RUN_DIRECTORY / "run.json",
        {
            "repository": GITHUB_URL,
            "commit": SOURCE_COMMIT,
            **INPUT_SETTINGS,
            "model": DEFAULT_CONFIG,
            "torch": str(torch.__version__),
            "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(),
        },
    )


def run_experiment(experiment, num_images):
    config = {
        **DEFAULT_CONFIG,
        "tau": 0 if experiment == "no_epg" else DEFAULT_CONFIG["tau"],
    }
    predictor.set_tau(config["tau"])
    return rep.evaluate(
        predictor,
        ENTRIES,
        CLASS_NAMES,
        RUN_DIRECTORY / experiment,
        experiment,
        num_images,
        settings={**INPUT_SETTINGS, "model": config},
        checkpoint_every=CHECKPOINT_EVERY,
        input_workers=INFERENCE_PREFETCH_WORKERS,
        progress=tqdm,
    )

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


构建 SAM 3 模型 |          | 0/4 [已用 00:00]

## 6. 流程检查

默认只跑前 20 张；子集 mIoU 不与论文完整验证集结果比较。
失败时停止并保存已完成进度，不跳过图片。

In [ ]:
# @title 运行少量图片检查流程 { display-mode: "form" }
run_experiment("smoke", SMOKE_IMAGES)

smoke:   0%|          | 0/20 [00:00<?, ?张/s]

smoke: 20 张，mIoU=20.05


{'experiment': 'smoke',
 'num_images': 20,
 'mIoU': 20.05,
 'mAcc': 53.95,
 'aAcc': 71.81,
 'per_class': [{'class_id': 0,
   'name': 'wall',
   'IoU': 40.17,
   'Acc': 45.29,
   'area_intersect': 255384,
   'area_union': 635756,
   'area_pred': 327227,
   'area_label': 563913},
  {'class_id': 1,
   'name': 'building',
   'IoU': 47.72,
   'Acc': 54.01,
   'area_intersect': 488169,
   'area_union': 1023028,
   'area_pred': 607314,
   'area_label': 903883},
  {'class_id': 2,
   'name': 'sky',
   'IoU': 97.58,
   'Acc': 98.12,
   'area_intersect': 742794,
   'area_union': 761214,
   'area_pred': 746991,
   'area_label': 757017},
  {'class_id': 3,
   'name': 'floor',
   'IoU': 72.19,
   'Acc': 82.47,
   'area_intersect': 241928,
   'area_union': 335134,
   'area_pred': 283716,
   'area_label': 293346},
  {'class_id': 4,
   'name': 'tree',
   'IoU': 56.7,
   'Acc': 92.96,
   'area_intersect': 84092,
   'area_union': 148322,
   'area_pred': 141955,
   'area_label': 90459},
  {'class_id': 5,
 

## 7. 完整验证集

计算完整 2,000 张原图的全局 mIoU，论文参考值为 **40.03%**。
输出总体指标、逐类指标和逐图耗时。

In [ ]:
# @title 默认配置：完整 2,000 张 { display-mode: "form" }
run_experiment("full", 2000)

full:   0%|          | 0/2000 [00:00<?, ?张/s]

full: 2000 张，mIoU=39.98


{'experiment': 'full',
 'num_images': 2000,
 'mIoU': 39.98,
 'mAcc': 60.33,
 'aAcc': 75.02,
 'per_class': [{'class_id': 0,
   'name': 'wall',
   'IoU': 74.71,
   'Acc': 80.67,
   'area_intersect': 63397100,
   'area_union': 84862240,
   'area_pred': 69675592,
   'area_label': 78583776},
  {'class_id': 1,
   'name': 'building',
   'IoU': 57.47,
   'Acc': 63.15,
   'area_intersect': 32450072,
   'area_union': 56467168,
   'area_pred': 37527616,
   'area_label': 51389616},
  {'class_id': 2,
   'name': 'sky',
   'IoU': 93.48,
   'Acc': 95.38,
   'area_intersect': 37950524,
   'area_union': 40597520,
   'area_pred': 38760404,
   'area_label': 39787648},
  {'class_id': 3,
   'name': 'floor',
   'IoU': 78.96,
   'Acc': 84.7,
   'area_intersect': 26305548,
   'area_union': 33316498,
   'area_pred': 28564690,
   'area_label': 31057338},
  {'class_id': 4,
   'name': 'tree',
   'IoU': 72.57,
   'Acc': 88.76,
   'area_intersect': 18766172,
   'area_union': 25858356,
   'area_pred': 23482208,
   'a

## 8. 关闭 EPG 的完整对照

仅将 `tau` 改为 0，完整解码所有提示，其余作者配置、权重和签名不变。
同一套 2,000 张图再运行一次，论文参考值为 **40.05% mIoU**，通常比默认配置慢。
第 2 节的 `RUN_NO_EPG=False` 可跳过本组。

**中断：** 点击 VS Code 的“中断执行”或 Colab 停止按钮。
已完成图片会写入断点，当前未完成图片不计数；等待当前 CUDA 调用收尾后，
重跑对应实验单元格继续。程序在收尾期间会拒绝新推理和重建模型。
保存断点时不要连续点击停止；强制重启可能丢失尚未写完的断点。

In [9]:
# @title 可选：关闭 EPG 的完整 2,000 张 { display-mode: "form" }
if RUN_NO_EPG:
    run_experiment("no_epg", 2000)
else:
    print("已跳过关闭 EPG 的对照。")

no_epg:   0%|          | 0/2000 [00:00<?, ?张/s]

no_epg: 2000 张，mIoU=40.07


## 9. 输出对照结果

各实验保存 `results.json`、`per_class.csv`、`per_image.csv` 和一个 `checkpoint.npz`。
完整组结束后生成 `comparison.csv/json`，对照论文 mIoU。
`inference_fps` 统计模型推理、同步和标签传回 CPU；`pipeline_fps` 还包含等待输入和指标计算，
不重复计入与 GPU 重叠的预取时间，均不含断点写盘，均不同于论文 Runner FPS。
续跑结果会标记已恢复的图片数量；跨会话混合计时不适合严格比较速度。

In [10]:
# @title 生成论文结果对照表 { display-mode: "form" }
import pandas as pd
from IPython.display import display

experiments = ["full"] + (["no_epg"] if RUN_NO_EPG else [])
results = [rep.read_json(RUN_DIRECTORY / name / "results.json") for name in experiments]
if any(result is None or result["num_images"] != 2000 for result in results):
    raise RuntimeError("请先完成本次需要的完整验证集实验。")
display(pd.DataFrame(rep.write_comparison(RUN_DIRECTORY, results)))
print("结果目录：", RUN_DIRECTORY)

,experiment,num_images,mIoU,paper_mIoU,difference_from_paper_pp,inference_fps,pipeline_fps,resumed_from_images
0,full,2000,39.98,40.03,-0.05,0.744013,0.741006,0
1,no_epg,2000,40.07,40.05,0.02,0.199419,0.199202,0


结果目录： /content/drive/MyDrive/results/ActiveSAM_reproduction/20261121_111812


## 10. 释放当前连接的 Colab 虚拟机

用于一键运行后自动释放 Colab 资源，节省运行额度。

In [11]:
# @title 释放 GPU 资源 { display-mode: "form" }
from google.colab import runtime

runtime.unassign()